### Google ADK (and a quick look at A2A)

In [1]:
# Importing Libraries and Setup
import os 
from dotenv import load_dotenv 
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from quiet import silence
silence() # Turn down some noisy library logging so the agent's trace is easy to read

load_dotenv(override=True)

True

### Step 1: Create the agent
In ADK an agent is an `LlmAgent`: a model plus an `instruction`, which is its system prompt. That is the whole object, no wrapper class and no registration step.

In [2]:
MODEL = 'gemini-3.5-flash-lite'

agent = LlmAgent(
    model=MODEL,
    name="assistant",
    instruction="You are a concise, friendly assistant. Reply in a single short sentence."
)

### Step 2: Run it  
`InMemoryRunner(...).run_debug(..., verbose=True)` runs the agent and prints each step, so you can watch what it does. With no tools yet there is nothing to loop over, so the agent just replies. This is still only an LLM call.

In [4]:
result = await InMemoryRunner(agent=agent).run_debug("Say hello in Marathi.", verbose=True)

assistant > नमस्कार! (Namaskar!)


### Our project: a SQLite todo board

The worker coordinates through a tiny SQLite board: one file, one table, no server to run.  
`board.py` is a handful of small functions.  
A worker is handed one **goal**; to reach it, it writes its own **step** todos under that goal, ticks each one off as it goes, and closes the goal at the end.  
Under the hood the board is just a list of dicts (a goal has **parent_id** of None, a step points at its goal):

In [5]:
import board 

board.reset_board()
board.add_goal("Read notes.txt, translate its content into natural Marathi, and write the Marathi to marathi.txt.")
board.list_todos()

[{'id': 1,
  'parent_id': None,
  'task': 'Read notes.txt, translate its content into natural Marathi, and write the Marathi to marathi.txt.',
  'status': 'pending',
  'result': ''}]

`show_board()` prints that same data nicely, in the rich style from Week 1: each goal with its steps indented beneath, done tasks struck through in green and the one in progress in yellow. There are no steps yet; the worker writes its own when it plans.

In [6]:
board.show_board()

Goal # 1: Read notes.txt, translate its content into natural Marathi, and write the Marathi to marathi.txt.

### Step 3: Add tools

A tool in ADK is just a typed Python function with a docstring. The docstring becomes the description the model reads, the type hints become the argument schema, and you hand the function to the agent in its `tools=[...]` list.

We write three small board tools: `show_todos` to read the board, `plan_steps` to break a goal into steps, and `complete_task` to mark a todo done. Here we give a quick agent just two of them and ask what is on the board; watch it decide, on its own, to call `show_todos` before it answers. That decide, call, read, answer is the agent loop starting to turn. All three tools come together in step 5.

In [7]:
def show_todos() -> list[dict]:
    """List every todo in the board. A goal has parent_id None; a step has parent_id set to its goal's id"""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into a ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

In [8]:
board_agent = LlmAgent(
    model=MODEL,
    name='board_agent',
    instruction="You help manange the shared todo board.",
    tools=[show_todos, plan_steps, complete_task]
)

result = await InMemoryRunner(agent=board_agent).run_debug("What is on the board right row, and what it its status?", verbose=True)

d:\Projects\agent_projects\.venv\Lib\site-packages\google\adk\models\llm_request.py:306: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


board_agent > [Calling tool: show_todos({})]
board_agent > [Tool result: {'result': [{'id': 1, 'parent_id': None, 'task': 'Read notes.txt, translate its content into natural...]
board_agent > On the board right now, there is one goal:

* **Task:** Read notes.txt, translate its content into natural Marathi, and write the Marathi to marathi.txt.
* **Status:** Pending (not yet completed)


### A tool can reach the real world too

The board tools talk to a little database. A tool can just as easily reach outside the machine. Here is one that sends a push notification to your phone through Pushover. It is the same `send_push_notification` used elsewhere in the course, and it is still just a typed function with a docstring; ADK reads those to learn how to call it.

In [9]:
import requests

def send_push_notification(message: str) -> str:
    """Send a short push notification to the user's phone to report a result or that a job is done."""
    payload = {
        "token": os.getenv("PUSHOVER_TOKEN"),
        "user": os.getenv("PUSHOVER_USER"),
        "message": message,
        "sound": "climb"
    }
    response = requests.post("https://api.pushover.net/1/messages.json", data=payload)
    return f"Push notification sent (status {response.status_code})."

In [10]:
notifier = LlmAgent(
    model=MODEL,
    name="notifier",
    instruction="You notify the user. When asked, use your tool to send a push notification.",
    tools=[send_push_notification],
)

result = await InMemoryRunner(agent=notifier).run_debug("Send a push notification that says hello from Google ADK.", verbose=True)

notifier > [Calling tool: send_push_notification({'message': 'hello from Google ADK'})]
notifier > [Tool result: {'result': 'Push notification sent (status 200).'}]
notifier > I have sent the push notification with the message "hello from Google ADK".


### Step 4: Add MCP

MCP is just more tools: ones you did not write, connected over a small protocol. We give the agent the filesystem reference server (the same Node server every framework uses this week), scoped to a single `workspace` folder so it can only touch files in there. In ADK that is one `McpToolset` added to the same `tools=[...]` list.

ADK starts the server for you as a small Node process over `npx`. One detail worth knowing: we pass `errlog=subprocess.DEVNULL`. That quiets the server's startup logging, and it is also what lets the server run from a Jupyter kernel on Windows, where the kernel's stderr has no real file descriptor for the server to write to. On Mac and Linux it changes nothing.

In [11]:
import subprocess
from pathlib import Path
from google.adk.tools.mcp_tool import McpToolset, StdioConnectionParams
from mcp import StdioServerParameters

workspace = Path("task_worker/workspace").resolve()   # the only folder the agent may touch
workspace.mkdir(parents=True, exist_ok=True)

filesystem = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="npx",
            args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
            cwd=str(workspace),  # start the server in the workspace so relative file names resolve there
        ),
        timeout=60
    ),
    errlog=subprocess.DEVNULL,
)

d:\Projects\agent_projects\.venv\Lib\site-packages\google\adk\features\_feature_decorator.py:71: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [12]:
file_agent = LlmAgent(
    model=MODEL,
    name="file_agent",
    instruction="You can read and write files in your workspace. Use your tools to do what is asked.",
    tools=[filesystem],
)

In [13]:
result = await InMemoryRunner(agent=file_agent).run_debug("Read notes.txt and summarize it in one short sentence.", verbose=True)

d:\Projects\agent_projects\.venv\Lib\site-packages\google\adk\tools\mcp_tool\mcp_toolset.py:407: UserWarning: [EXPERIMENTAL] feature FeatureName._MCP_GRACEFUL_ERROR_HANDLING is enabled.
  session = await self._mcp_session_manager.create_session(
d:\Projects\agent_projects\.venv\Lib\site-packages\google\adk\features\_feature_decorator.py:71: UserWarning: [EXPERIMENTAL] feature FeatureName.BASE_AUTHENTICATED_TOOL is enabled.
  check_feature_enabled()


file_agent > [Calling tool: list_allowed_directories({})]
file_agent > [Tool result: {'content': [{'type': 'text', 'text': 'Allowed directories:\nD:\\Projects\\agent_projects\\5_agent_f...]
file_agent > [Calling tool: read_text_file({'path': 'notes.txt'})]
file_agent > [Tool result: {'content': [{'type': 'text', 'text': 'Welcome to the team.\r\n\r\nToday we are building a small lan...]
file_agent > Team members are instructed to carefully pick, complete, and mark off individual tasks from the shared board to build a small language tutor.


### Step 5: Put it in a loop with a goal

Now the payoff. Give one agent all three board tools and the filesystem server, hand it the goal, and let it run. It plans its own steps on the board, works them with its file tools, ticks each one off, and closes the goal when the work is done. That is the agent loop running on its own: read, plan, act, check off, repeat.

This time we drive the loop with `run_async` rather than `run_debug`, stepping through its events ourselves. We print each tool call as the agent makes it, and re-draw the board whenever a step is planned or ticked off, so you watch the plan appear and then strike through one line at a time.

In [15]:
from google.genai import types

INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into,
added to the board under the goal. Then carrum them out with your file tools. marking each step done as you finish it. Once the steps are 
all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

worker = LlmAgent(
    model=MODEL,
    name='task_worker',
    instruction=INSTRUCTIONS,
    tools = [show_todos, plan_steps, complete_task, filesystem, send_push_notification]
)

board.reset_board()
goal_id = board.add_goal("Read notes.txt, translate its content into Marathi, and write the Marathi to marathi.txt and send a push notification with a summary.")
board.claim_todo(goal_id)

# Drive the loop ourselves with run_async instead of run_debug, so we can watch it:
# print each tool call, and re-draw the board whenever a step is planned or ticked off.
runner = InMemoryRunner(agent=worker)
session = await runner.session_service.create_session(app_name=runner.app_name, user_id='sada')

async for event in runner.run_async(
    user_id='sada',
    session_id=session.id,
    new_message=types.UserContent("Please work the pending goal on the board"),
):
    for call in event.get_function_calls():
        print(f"tool: {call.name}({call.args})")
    if any(r.name in ("plan_steps", "complete_task") for r in event.get_function_responses()):
        board.show_board()
    if event.is_final_response():
        print(event.content.parts[0].text)

tool: show_todos({})
tool: plan_steps({'steps': ['Read notes.txt to understand its content', 'Translate the content of notes.txt into Marathi', 'Write the translated Marathi text into marathi.txt', 'Send a push notification with a summary of the work done'], 'goal_id': 1})


Goal # 1: Read notes.txt, translate its content into Marathi, and write the Marathi to marathi.txt and send a push notification with a summary.
 Step # 2: Read notes.txt to understand its content
 Step # 3: Translate the content of notes.txt into Marathi
 Step # 4: Write the translated Marathi text into marathi.txt
 Step # 5: Send a push notification with a summary of the work done

tool: show_todos({})
tool: read_text_file({'path': 'notes.txt'})
tool: complete_task({'result': "Successfully read notes.txt content: 'Welcome to the team. Today we are building a small language tutor...'", 'task_id': 2})


Goal # 1: Read notes.txt, translate its content into Marathi, and write the Marathi to marathi.txt and send a push notification with a summary.
 Step # 2: Read notes.txt to understand its content Successfully read notes.txt content: 'Welcome to the team. Today we are building a small language tutor...'
 Step # 3: Translate the content of notes.txt into Marathi
 Step # 4: Write the translated Marathi text into marathi.txt
 Step # 5: Send a push notification with a summary of the work done

tool: complete_task({'task_id': 3, 'result': 'Successfully translated notes.txt content into Marathi.'})


...

tool: send_push_notification({'message': 'Successfully translated notes.txt into Marathi and saved it to marathi.txt.'})
tool: complete_task({'result': 'Successfully sent push notification with summary.', 'task_id': 5})


Goal # 1: Read notes.txt, translate its content into Marathi, and write the Marathi to marathi.txt and send a push notification with a summary.
 Step # 2: Read notes.txt to understand its content Successfully read notes.txt content: 'Welcome to the team. Today we are building a small language tutor...'
 Step # 3: Translate the content of notes.txt into Marathi Successfully translated notes.txt content into Marathi.
 Step # 4: Write the translated Marathi text into marathi.txt Successfully wrote Marathi translation to marathi.txt
 Step # 5: Send a push notification with a summary of the work done Successfully sent push notification with summary.

tool: complete_task({'result': 'Completed all steps: read notes.txt, translated to Marathi, wrote to marathi.txt, and sent push notification.', 'task_id': 1})


Goal # 1: Read notes.txt, translate its content into Marathi, and write the Marathi to marathi.txt and send a push notification with a summary. Completed all steps: read notes.txt, translated to Marathi, wrote to marathi.txt, and sent push notification.
 Step # 2: Read notes.txt to understand its content Successfully read notes.txt content: 'Welcome to the team. Today we are building a small language tutor...'
 Step # 3: Translate the content of notes.txt into Marathi Successfully translated notes.txt content into Marathi.
 Step # 4: Write the translated Marathi text into marathi.txt Successfully wrote Marathi translation to marathi.txt
 Step # 5: Send a push notification with a summary of the work done Successfully sent push notification with summary.

tool: show_todos({})
I have successfully completed all the steps for the pending goal:
1. Read `notes.txt`.
2. Translated the content into Marathi.
3. Wrote the translated text into `marathi.txt`.
4. Sent a push notification reporting the successful completion.
5. Closed the goal on the shared board.
